# Tweets Feature Engineering

## Memory Debug log

In [2]:
import torch
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

0.0 GB allocated
0.0 GB reserved


In [3]:
def gpu_status():
	print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
	print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
	print(f"Free/Total: {torch.cuda.mem_get_info()[0]/1e9:.2f} / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

gpu_status()

Allocated: 0.00 GB
Reserved:  0.00 GB
Free/Total: 7.37 / 8.52 GB


## Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np
import re
import emoji
import ast
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers import pipeline
import gc

In [ ]:
def _as_list(value):
	if value is None:
		return []
	if isinstance(value, np.ndarray):
		return value.tolist()
	if isinstance(value, list):
		return value
	return []

REF_TIME = pd.Timestamp("2022-04-01", tz="UTC")

## Load Dataset

In [6]:
df_tweets = pd.read_parquet("datasets/final_outputs/df_tweets_final.parquet")
df_users = pd.read_parquet("./datasets/final_outputs/df_users_final.parquet")

## Reformat tweet data to an older version to match code

In [ ]:
df_flat = df_tweets.copy()

if "tweets" in df_flat.columns and "text" not in df_flat.columns:
	df_flat = df_flat.rename(columns={"tweets": "text"})
if "tweet_id" in df_flat.columns and "id" not in df_flat.columns:
	df_flat = df_flat.rename(columns={"tweet_id": "id"})
if "author_id" in df_flat.columns:
	assert df_flat["author_id"].notna().all(), "author_id has nulls; ids would become floats"
	s = df_flat["author_id"].astype("int64").astype("string")
	df_flat["author_id"] = s.where(s.str.startswith("u"), "u" + s)

df_flat["user_id"] = df_flat["author_id"]

TWEET_FEATURE_FIELDS = [
	"id", "text", "created_at", "lang",
	"entities", "attachments", "public_metrics", "referenced_tweets",
	"in_reply_to_user_id", "geo", "possibly_sensitive", "reply_settings",
]

tweet_dict_cols = [c for c in TWEET_FEATURE_FIELDS if c in df_flat.columns]
missing = set(TWEET_FEATURE_FIELDS) - set(tweet_dict_cols)
if missing:
	print(f"WARNING: expected fields not present: {sorted(missing)}")
print(f"Building tweet dicts from {len(tweet_dict_cols)} fields "
      f"(of {len(df_flat.columns)} available)")

df_flat = df_flat[["user_id"] + tweet_dict_cols]
df_flat["tweet_dict"] = df_flat[tweet_dict_cols].to_dict(orient="records")

df_tweets_grouped = (
	df_flat.groupby("user_id")["tweet_dict"]
	.apply(list)
	.reset_index(name="tweets")
	.rename(columns={"user_id": "id"})
)

print(df_tweets_grouped.shape)
df_tweets_grouped.head()

del df_flat
gc.collect()

Building tweet dicts from 12 fields (of 18 available)
(38771, 2)


0

## Data Inspection

In [8]:
df_tweets = df_tweets_grouped
df_tweets.head(10)

,id,tweets
0,u1000050990,"[{'id': 't1491853123429847048', 'text': '@anto..."
1,u1000054188183904257,"[{'id': 't1491355901462474753', 'text': 'RT @Z..."
2,u100008784,"[{'id': 't1499241032160096256', 'text': 'My ty..."
3,u1000156482690838530,"[{'id': 't1491545672713187332', 'text': 'Jenni..."
4,u1000230050736701441,"[{'id': 't1492039865315500034', 'text': '@Spik..."
5,u1000242837852688385,"[{'id': 't1499748737782013955', 'text': 'I've ..."
6,u100024370,"[{'id': 't1502729327170887680', 'text': 'Right..."
7,u1000288828194615296,"[{'id': 't1497540902788612099', 'text': 'RT @b..."
8,u1000295179,"[{'id': 't1491893683440041987', 'text': '@aims..."
9,u1000298716006252544,"[{'id': 't1491517068524474369', 'text': 'RT @P..."


In [ ]:
df_tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38771 entries, 0 to 38770
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      38771 non-null  string
 1   tweets  38771 non-null  object
dtypes: object(1), string(1)
memory usage: 605.9+ KB


In [ ]:
df_users.head()

,id,label,split,participation_degree,created_at,description,entities,location,name,pinned_tweet_id,...,public_metrics.tweet_count,public_metrics.listed_count,entities.url.urls,entities.description,entities.url,entities.description.urls,entities.description.mentions,entities.description.hashtags,entities.description.cashtags,withheld.country_codes
0,u2664730894,human,train,1726,2014-07-02 17:56:46+00:00,creative _,NaN,🎈,olawale 💨,NaN,...,1823,0,None,NaN,NaN,None,None,None,None,None
1,u1679822588,bot,train,1127,2013-08-18 04:21:48+00:00,,NaN,🇬🇧,Grian,1.143808e+18,...,1400,448,"[{'display_url': 'youtube.com/c/grian', 'end':...",NaN,NaN,None,None,None,None,None
2,u2465283662,bot,test,3346,2014-04-27 00:20:12+00:00,"paper tweets, dms are open",NaN,None,AK,NaN,...,9194,605,None,NaN,NaN,None,None,None,None,None
3,u1467973039883182090,human,train,1207,2021-12-06 21:44:04+00:00,https://t.co/Hmg5gBvd9A,NaN,None,صارا,NaN,...,146,2,None,NaN,NaN,"[{'display_url': 't.me/BiChatBot?star…', 'end'...",None,None,None,None
4,u234059290,human,train,2267,2011-01-04 19:11:39+00:00,Come for the science (genetics & cell biology)...,NaN,"Salt Lake City,UT, USA",Professor Booty PhD,1.470252e+18,...,91381,116,"[{'display_url': 'profbootyphd.wordpress.com',...",NaN,NaN,None,None,"[{'end': 129, 'start': 112, 'tag': 'BlackLives...",None,None


In [ ]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   id                              40000 non-null  object  
 1   label                           40000 non-null  category
 2   split                           40000 non-null  category
 3   participation_degree            40000 non-null  int64   
 4   created_at                      40000 non-null  object  
 5   description                     40000 non-null  object  
 6   entities                        0 non-null      float64 
 7   location                        28521 non-null  object  
 8   name                            40000 non-null  object  
 9   pinned_tweet_id                 18253 non-null  float64 
 10  profile_image_url               40000 non-null  object  
 11  protected                       40000 non-null  bool    
 12  url               

## Tweets Inspection

In [ ]:
df_tweets["num_tweets"] = df_tweets["tweets"].str.len()
df_tweets["has_tweets"] = 1

print(f"Users in grouped frame: {len(df_tweets):,}")
print(df_tweets["num_tweets"].describe())

Users in grouped frame: 38,771
count    38771.000000
mean       243.409584
std        274.079036
min          1.000000
25%         57.000000
50%        200.000000
75%        280.000000
max       4506.000000
Name: num_tweets, dtype: float64


## Feature Engineering

### User Activity Features

In [13]:
# whether or not user has tweets
# df_tweets["has_tweets"] = df_tweets["tweets"].apply(lambda x: len(x) != 0).astype(int)

# # number tweets
# df_tweets["num_tweets"] = df_tweets["tweets"].apply(lambda x: len(x))

In [ ]:
def to_datetime_safe(value):
	if value is None:
		return None
	if isinstance(value, str):
		return datetime.fromisoformat(value.replace('Z', '+00:00'))
	if isinstance(value, pd.Timestamp):
		value = value.to_pydatetime()
	if isinstance(value, datetime):
		if value.tzinfo is None:
			from datetime import timezone
			value = value.replace(tzinfo=timezone.utc)
		return value
	raise TypeError(f"Unrecognized timestamp type: {type(value)}")

In [ ]:
created_at_map = df_users.set_index('id')['created_at'].to_dict()
assert len(set(created_at_map) & set(df_tweets['id'])) > 0, "id spaces do not match"

def compute_tweet_frequency(row):
	tweet_list = row['tweets']
	user_id = row['id']

	num_tweets = row['num_tweets']

	if num_tweets == 0:
		return 0

	created_at_str = created_at_map.get(user_id)

	if created_at_str is None:
		return 0

	created_at = to_datetime_safe(created_at_str)

	account_age_days = (REF_TIME.astimezone(created_at.tzinfo) - created_at).days

	if account_age_days <= 0:
		return 0

	return num_tweets / account_age_days

df_tweets['tweets_frequency'] = df_tweets.apply(compute_tweet_frequency, axis=1)

In [ ]:

def compute_posting_regularity(tweet_list):
	timestamps = []

	for t in tweet_list:
		if 'created_at' in t and t['created_at'] is not None:
			timestamps.append(to_datetime_safe(t['created_at']))

	if len(timestamps) <= 2:
		return 0

	timestamps.sort()

	diffs = [
		(timestamps[i] - timestamps[i-1]).total_seconds() / (60*60*24)
		for i in range(1, len(timestamps))
	]

	return np.var(diffs)

### Content Features

In [ ]:
def compute_text_length(tweet_list):
	lengths = []
	word_counts = []
	word_lengths = []

	if not isinstance(tweet_list, list):
		return pd.Series([0, 0, 0])

	for tweet in tweet_list:
		if not isinstance(tweet, dict):
			continue

		text = tweet.get("text")

		if pd.isna(text):
			continue

		text = str(text).strip()
		if text == "":
			continue

		char_len = len(text)
		lengths.append(char_len)

		words = re.findall(r"\w+", text)
		word_counts.append(len(words))

		if len(words) > 0:
			word_lengths.append(sum(len(w) for w in words) / len(words))

	if len(lengths) == 0:
		return pd.Series([0, 0, 0])

	avg_length = sum(lengths) / len(lengths)
	avg_words = sum(word_counts) / len(word_counts)
	avg_word_length = sum(word_lengths) / len(word_lengths) if len(word_lengths) > 0 else 0

	return pd.Series([avg_length, avg_words, avg_word_length])

In [ ]:
def compute_text_style(tweet_list):
	uppercase_pct = []
	digit_pct = []
	special_char_pct = []
	emoji_count = []

	if not isinstance(tweet_list, list):
		return pd.Series([0, 0, 0, 0])

	for tweet in tweet_list:
		if not isinstance(tweet, dict):
			continue

		text = tweet.get('text')

		if pd.isna(text):
			continue

		text = str(text)
		total_chars = len(text)

		if total_chars == 0:
			continue

		uppercase_count = sum(1 for c in text if c.isupper())
		uppercase_pct.append(uppercase_count / total_chars)

		digit_count = sum(1 for c in text if c.isdigit())
		digit_pct.append(digit_count / total_chars)

		special_chars = sum(
			1 for c in text if not c.isalnum() and c not in [' ', '#', '@']
		)
		special_char_pct.append(special_chars / total_chars)

		emoji_count.append(len(emoji.emoji_list(text)))

	if len(uppercase_pct) == 0:
		return pd.Series([0, 0, 0, 0])

	return pd.Series([
		sum(uppercase_pct) / len(uppercase_pct),
		sum(digit_pct) / len(digit_pct),
		sum(special_char_pct) / len(special_char_pct),
		sum(emoji_count) / len(emoji_count)
	])

In [ ]:
def compute_hashtag(tweet_list):
	hashtags_per_tweet = []
	unique_hashtags_per_tweet = []
	has_hashtag_flags = []

	if not isinstance(tweet_list, list):
		return pd.Series([0, 0, 0])

	for tweet in tweet_list:
		if not isinstance(tweet, dict):
			continue

		entities = tweet.get("entities")
		if not isinstance(entities, dict):
			continue

		hashtags = _as_list(entities.get("hashtags"))

		hashtag_tags = []
		for h in hashtags:
			if isinstance(h, dict):
				if "tag" in h:
					hashtag_tags.append(str(h["tag"]))
				elif "text" in h:
					hashtag_tags.append(str(h["text"]))

		hashtags_per_tweet.append(len(hashtag_tags))
		unique_hashtags_per_tweet.append(len(set(hashtag_tags)))
		has_hashtag_flags.append(1 if len(hashtag_tags) > 0 else 0)

	if len(hashtags_per_tweet) == 0:
		return pd.Series([0, 0, 0])

	avg_hashtags = sum(hashtags_per_tweet) / len(hashtags_per_tweet)
	avg_unique_hashtags = sum(unique_hashtags_per_tweet) / len(unique_hashtags_per_tweet)
	pct_tweets_with_hashtags = sum(has_hashtag_flags) / len(has_hashtag_flags)

	overuse_hashtags = 1 if avg_unique_hashtags < avg_hashtags else 0

	return pd.Series([avg_hashtags, overuse_hashtags, pct_tweets_with_hashtags])

In [ ]:
def compute_mention(tweet_list):
	mentions_per_tweet = []
	has_mentions_flags = []

	if not isinstance(tweet_list, list):
		return pd.Series([0, 0])

	for tweet in tweet_list:
		if not isinstance(tweet, dict):
			continue

		entities = tweet.get("entities")
		if not isinstance(entities, dict):
			continue

		mentions = _as_list(entities.get("mentions")) or _as_list(entities.get("user_mentions"))

		mentions_per_tweet.append(len(mentions))
		has_mentions_flags.append(1 if len(mentions) > 0 else 0)

	if len(mentions_per_tweet) == 0:
		return pd.Series([0, 0])

	avg_mentions = sum(mentions_per_tweet) / len(mentions_per_tweet)
	pct_tweets_with_mentions = sum(has_mentions_flags) / len(has_mentions_flags)

	return pd.Series([avg_mentions, pct_tweets_with_mentions])

In [ ]:
def compute_url(tweet_list):
	urls_per_tweet = []
	has_urls_flags = []

	if not isinstance(tweet_list, list):
		return pd.Series([0, 0])

	for tweet in tweet_list:
		if not isinstance(tweet, dict):
			continue

		entities = tweet.get("entities")
		if not isinstance(entities, dict):
			continue

		urls = _as_list(entities.get("urls"))

		urls_per_tweet.append(len(urls))
		has_urls_flags.append(1 if len(urls) > 0 else 0)

	if len(urls_per_tweet) == 0:
		return pd.Series([0, 0])

	avg_urls = sum(urls_per_tweet) / len(urls_per_tweet)
	pct_tweets_with_urls = sum(has_urls_flags) / len(has_urls_flags)

	return pd.Series([avg_urls, pct_tweets_with_urls])

In [ ]:
def compute_attachment(tweet_list):
	attachments_per_tweet = []
	has_attachments_flags = []

	if not isinstance(tweet_list, list):
		return pd.Series([0, 0])

	for tweet in tweet_list:
		if not isinstance(tweet, dict):
			continue

		attachments = tweet.get("attachments")
		if not isinstance(attachments, dict):
			continue

		n = len(_as_list(attachments.get("media_keys")))

		attachments_per_tweet.append(n)
		has_attachments_flags.append(1 if n > 0 else 0)

	if len(attachments_per_tweet) == 0:
		return pd.Series([0, 0])

	avg_attachments = sum(attachments_per_tweet) / len(attachments_per_tweet)
	pct_tweets_with_attachments = sum(has_attachments_flags) / len(has_attachments_flags)

	return pd.Series([avg_attachments, pct_tweets_with_attachments])

In [ ]:
def compute_sensitivity(tweet_list):
	sensitivity_per_tweet = []

	for tweet in tweet_list:
		sensitivity = tweet.get('possibly_sensitive', False)
		sensitivity_per_tweet.append(1 if sensitivity else 0)

	if len(sensitivity_per_tweet) == 0:
		return 0
	else:
		return sum(sensitivity_per_tweet) / len(sensitivity_per_tweet)

### Sentiment Analysis

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

model_name = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

sentiment_pipeline = pipeline(
	"sentiment-analysis",
	model=model,
	tokenizer=tokenizer,
	device=device
)

all_texts = []
tweet_counts = []

MAX_TWEETS_PER_USER = 20
MAX_CHARS = 280

for tweet_list in df_tweets["tweets"]:
	texts = []

	if isinstance(tweet_list, list):
		for t in tweet_list[-MAX_TWEETS_PER_USER:]:
			if not isinstance(t, dict):
				continue

			text = t.get("text")
			if pd.isna(text):
				continue

			text = str(text).strip()
			if text == "":
				continue

			texts.append(text[:MAX_CHARS])

	all_texts.extend(texts)
	tweet_counts.append(len(texts))

print("Total texts to score:", len(all_texts))

all_results = []
CHUNK_SIZE = 10000
BATCH_SIZE = 64

order = sorted(range(len(all_texts)), key=lambda i: len(all_texts[i]))
sorted_texts = [all_texts[i] for i in order]

sorted_results = []
for i in range(0, len(sorted_texts), CHUNK_SIZE):
	print(f"Processing {i} to {min(i + CHUNK_SIZE, len(sorted_texts))} ...")
	chunk_results = sentiment_pipeline(
		sorted_texts[i:i + CHUNK_SIZE],
		batch_size=BATCH_SIZE,
		truncation=True,
		max_length=128,
		padding=True,
	)
	sorted_results.extend(chunk_results)

	torch.cuda.empty_cache()
	gc.collect()

all_results = [None] * len(all_texts)
for pos, res in zip(order, sorted_results):
	all_results[pos] = res

mapped_sentiments = []

for r in all_results:
	label = r["label"]
	if label == "LABEL_2":
		mapped_sentiments.append("positive")
	elif label == "LABEL_0":
		mapped_sentiments.append("negative")
	else:
		mapped_sentiments.append("neutral")

idx = 0
features = []

for count in tweet_counts:
	if count == 0:
		features.append([0, 0, 0])
		continue

	subset = mapped_sentiments[idx: idx + count]
	idx += count

	total = len(subset)
	pos = subset.count("positive") / total
	neg = subset.count("negative") / total
	neu = subset.count("neutral") / total

	features.append([pos, neg, neu])

df_tweets[["avg_positive", "avg_negative", "avg_neutral"]] = pd.DataFrame(
	features,
	index=df_tweets.index
)

print(df_tweets[["avg_positive", "avg_negative", "avg_neutral"]].head())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Using GPU
Total texts to score: 758853
Processing 0 to 10000 ...
Processing 10000 to 20000 ...
Processing 20000 to 30000 ...
Processing 30000 to 40000 ...
Processing 40000 to 50000 ...
Processing 50000 to 60000 ...
Processing 60000 to 70000 ...
Processing 70000 to 80000 ...
Processing 80000 to 90000 ...
Processing 90000 to 100000 ...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processing 100000 to 110000 ...
Processing 110000 to 120000 ...
Processing 120000 to 130000 ...
Processing 130000 to 140000 ...
Processing 140000 to 150000 ...
Processing 150000 to 160000 ...
Processing 160000 to 170000 ...
Processing 170000 to 180000 ...
Processing 180000 to 190000 ...
Processing 190000 to 200000 ...
Processing 200000 to 210000 ...
Processing 210000 to 220000 ...
Processing 220000 to 230000 ...
Processing 230000 to 240000 ...
Processing 240000 to 250000 ...
Processing 250000 to 260000 ...
Processing 260000 to 270000 ...
Processing 270000 to 280000 ...
Processing 280000 to 290000 ...
Processing 290000 to 300000 ...
Processing 300000 to 310000 ...
Processing 310000 to 320000 ...
Processing 320000 to 330000 ...
Processing 330000 to 340000 ...
Processing 340000 to 350000 ...
Processing 350000 to 360000 ...
Processing 360000 to 370000 ...
Processing 370000 to 380000 ...
Processing 380000 to 390000 ...
Processing 390000 to 400000 ...
Processing 400000 to 410000 ...
Processi

### Interaction Features

In [ ]:
def compute_tweet_types(tweet_list):
	retweets = 0
	quotes = 0
	replies = 0
	originals = 0

	for tweet in tweet_list:
		ref = _as_list(tweet.get("referenced_tweets"))
		ref_types = [r.get("type") for r in ref if isinstance(r, dict)]
		reply_flag = pd.notna(tweet.get("in_reply_to_user_id"))

		is_reply = reply_flag or ("replied_to" in ref_types)
		is_retweet = "retweeted" in ref_types
		is_quote = "quoted" in ref_types

		if is_reply:
			replies += 1
		if is_retweet:
			retweets += 1
		if is_quote:
			quotes += 1
		if not (is_reply or is_retweet or is_quote):
			originals += 1

	num_tweets = len(tweet_list)

	if num_tweets == 0:
		return pd.Series([0, 0, 0, 0])
	else:
		return pd.Series([
				retweets / num_tweets,
				quotes / num_tweets,
				replies / num_tweets,
				originals / num_tweets
		])

### Engagement Features

In [ ]:
def compute_engagement(tweet_list):
	retweets_per_tweet = []
	replies_per_tweet = []
	likes_per_tweet = []
	quotes_per_tweet = []

	for tweet in tweet_list:
		metrics = tweet.get('public_metrics')

		if not metrics:
			continue

		if isinstance(metrics, str):
			try:
				metrics_dict = ast.literal_eval(metrics)
			except:
				continue
		else:
			metrics_dict = metrics

		retweets_per_tweet.append(metrics_dict.get('retweet_count') or 0)
		replies_per_tweet.append(metrics_dict.get('reply_count') or 0)
		likes_per_tweet.append(metrics_dict.get('like_count') or 0)
		quotes_per_tweet.append(metrics_dict.get('quote_count') or 0)

	n = len(likes_per_tweet)
	if n == 0:
		return pd.Series([0, 0, 0, 0, 0])

	avg_likes = sum(likes_per_tweet) / n
	avg_retweets = sum(retweets_per_tweet) / n
	avg_replies = sum(replies_per_tweet) / n
	avg_quotes = sum(quotes_per_tweet) / n

	engagement_rate = (
			sum(likes_per_tweet) +
			sum(retweets_per_tweet) +
			sum(replies_per_tweet) +
			sum(quotes_per_tweet)
	) / len(tweet_list)

	return pd.Series([avg_likes, avg_retweets, avg_replies, avg_quotes, engagement_rate])

In [ ]:
def compute_reply_settings(tweet_list):
	reply_settings_flags = []

	for tweet in tweet_list:
		reply_settings = tweet.get('reply_settings')

		reply_settings_flags.append(0 if reply_settings in (None, "everyone") else 1)

	if len(reply_settings_flags) == 0:
		return 0

	pct_reply_settings = sum(reply_settings_flags) / len(reply_settings_flags)

	return pct_reply_settings

In [ ]:
def compute_fc_reply(tweet_list):
	total_replies = 0
	count_allow_reply = 0

	if len(tweet_list) == 0:
			return 0

	for tweet in tweet_list:
		reply_settings = tweet.get('reply_settings')
		reply_settings_flag = 0 if reply_settings in (None, "everyone") else 1

		metrics = tweet.get('public_metrics')

		if not metrics:
			continue

		if isinstance(metrics, str):
			try:
				metrics_dict = ast.literal_eval(metrics)
			except:
				continue
		else:
			metrics_dict = metrics

		reply_count = metrics_dict.get('reply_count') or 0

		total_replies += reply_count
		count_allow_reply += reply_settings_flag

	if count_allow_reply == 0:
		return 0
	else:
		return total_replies / count_allow_reply

### Other Features

In [ ]:
def compute_language(tweet_list):
	langs = []

	for tweet in tweet_list:
		lang = tweet.get('lang')

		if not lang:
			continue

		langs.append(lang)

	if len(langs) == 0:
		return 0

	num_unique_langs = len(set(langs))

	return num_unique_langs

In [ ]:
def compute_geo(tweet_list):
	geo_flags = []

	for tweet in tweet_list:
		geo = tweet.get('geo')

		geo_flags.append(1 if geo is not None else 0)

	if len(geo_flags) == 0:
		return 0

	pct_geo = sum(geo_flags) / len(geo_flags)

	return pct_geo

### Suspicious Behaviour

In [ ]:
def count_inconsistencies(tweet_list):
	has_inconsistencies = 0
	count = 0

	for tweet in tweet_list:
		ref = _as_list(tweet.get("referenced_tweets"))
		ref_is_none = len(ref) == 0
		ref_types = [r.get("type") for r in ref if isinstance(r, dict)]

		reply_flag = pd.notna(tweet.get("in_reply_to_user_id"))
		is_reply_ref = "replied_to" in ref_types

		if (reply_flag and ref_is_none) or (not reply_flag and is_reply_ref):
			count += 1

	if (count != 0):
			has_inconsistencies = 1

	return pd.Series([has_inconsistencies, count])

In [ ]:
import time

FEATURE_SPECS = [
	(compute_posting_regularity, ["posting_regularity"]),
	(compute_text_length,        ["avg_length_per_tweet", "avg_words_per_tweet", "avg_word_length_per_tweet"]),
	(compute_text_style,         ["avg_uppercase_pct", "avg_digit_pct", "avg_special_char_pct", "avg_emoji"]),
	(compute_hashtag,            ["avg_hashtags_per_tweet", "overuse_hashtags", "pct_tweets_with_hashtags"]),
	(compute_mention,            ["avg_mentions_per_tweet", "pct_tweets_with_mentions"]),
	(compute_url,                ["avg_urls_per_tweet", "pct_tweets_with_urls"]),
	(compute_attachment,         ["avg_attachments_per_tweet", "pct_tweets_with_attachments"]),
	(compute_sensitivity,        ["avg_sensitive"]),
	(compute_tweet_types,        ["avg_retweets", "avg_quotes", "avg_replies", "avg_originals"]),
	(compute_engagement,         ["avg_likes_per_tweet", "avg_retweets_per_tweet",
	                              "avg_replies_per_tweet", "avg_quotes_per_tweet", "engagement_rate"]),
	(compute_reply_settings,     ["pct_allow_reply"]),
	(compute_fc_reply,           ["fc_reply"]),
	(compute_language,           ["num_unique_langs"]),
	(compute_geo,                ["pct_geo_tweets"]),
	(count_inconsistencies,      ["has_reply_inconsistency", "reply_inconsistency_count"]),
]

ALL_FEATURE_COLS = [c for _, cols in FEATURE_SPECS for c in cols]


def _unpack(out, n, fn_name):
	if n == 1:
		return [out]
	try:
		vals = list(out)
	except TypeError:
		raise TypeError(f"{fn_name} returned a scalar where {n} values were expected")
	if len(vals) != n:
		raise ValueError(f"{fn_name} returned {len(vals)} values, expected {n}")
	return vals


t0 = time.perf_counter()
rows = []
for tweet_list in df_tweets["tweets"]:
	row = []
	for fn, cols in FEATURE_SPECS:
		row.extend(_unpack(fn(tweet_list), len(cols), fn.__name__))
	rows.append(row)

df_tweets[ALL_FEATURE_COLS] = pd.DataFrame(
	rows, columns=ALL_FEATURE_COLS, index=df_tweets.index
)

print(f"{len(ALL_FEATURE_COLS)} features over {len(df_tweets):,} users "
      f"in {time.perf_counter() - t0:,.1f}s")

for c in ["overuse_hashtags", "has_reply_inconsistency"]:
	df_tweets[c] = df_tweets[c].astype(int)

33 features over 38,771 users in 1,319.6s


## Summary

Total number of features: 39 \
Number of numerical features: 36 \
Number of binary features: 3 -> `has_tweets`, `overuse_hashtags`, `has_reply_inconsistency`
*   User Activity
	*   whether or not user has tweets `has_tweets`
	*   total number of tweets `num_tweets`
	*   tweet frequency -> average number of tweets per day relative to age `tweet_frequency`
	*   posting regularity -> variance of time gaps between consecutive tweets (in days) `posting_regularity`
*   Content Features
	*   Text Length
		*   average length per tweet `avg_length_per_tweet`
		*   average number of words per tweet `avg_words_per_tweet`
		*   average word length per tweet `avg_word_length_per_tweet`
		*   NOTE: values are inaccurate as the text of each tweet is truncated
	*   Text Style
		*   average percentage of uppercase characters in a tweet `avg_uppercase_pct`
		*   average percentage of digits in a tweet `avg_digit_pct`
		*   average percentage of special characters in a tweet (excluding # and @) `avg_special_char_pct`
		*   average number of emojis per tweet `avg_emoji`
		*   NOTE: values are inaccurate as the text of each tweet is truncated
	*   Hashtags
		*   average number of hashtags per tweet `avg_hashtags_per_tweet`
		*   whether or not the same hashtag is used twice in a tweet `overuse_hashtags`
		*   percentage of tweets with hashtags `pct_tweets_with_hashtags`
	*   Mentions
		*   average number of mentions per tweet `avg_mentions_per_tweet`
		*   percentage of tweets with mentions `pct_tweets_with_mentions`
	*   URLs
		*   average number of URLs per tweet `avg_urls_per_tweet`
		*   percentage of tweets with URLs `pct_tweets_with_urls`
	*   Attachments
		*   average number of attachments per tweet `avg_attachments_per_tweet`
		*   percentage of tweets with attachments `pct_tweets_with_attachments`
	*   Sensitivity of tweet
		*   average number of sensitive tweets `avg_sensitive`
	*   Sentiment Analysis
		*   average number of positive tweets `avg_positive`
		*   average number of negative tweets `avg_negative`
		*   average number of neutral tweets `avg_neutral`
		*   NOTE: values are inaccurate as the text of each tweet is truncated
*   Interaction Features
	*   average number of retweets `avg_retweets`
	*   average number of quoted tweets `avg_quotes`
	*   average number of tweets that are replies to other users `avg_replies`
	*   average number of original tweets `avg_originals`
*   Engagement Features
	*   average number of likes per tweet `avg_likes_per_tweet`
	*   average number of retweets per tweet `avg_retweets_per_tweet`
	*   average number of replies per tweet `avg_replies_per_tweet`
	*   average number of quotes per tweet `avg_quotes_per_tweet`
	*   engagement rate `engagement_rate`
	*   percentage of tweets with reply settings enabled `pct_allow_reply`
	*   feature crossing between reply settings and number of replies `fc_reply`
*   Other Features
	*   total number of languages used `num_unique_langs`
	*   percentage of geo-enabled tweets `pct_geo_tweets`   
*   Suspicious behaviour
	*   whether or not user has reply inconsistencies `has_reply_inconsistency`
	*   number of reply inconsistencies -> count of inconsistent behaviour between in_reply_to_user_id and type of referenced_tweets `reply_inconsistency_count`



In [ ]:
df_tweets.head()

,id,tweets,num_tweets,has_tweets,tweets_frequency,avg_positive,avg_negative,avg_neutral,posting_regularity,avg_length_per_tweet,...,avg_retweets_per_tweet,avg_replies_per_tweet,avg_quotes_per_tweet,engagement_rate,pct_allow_reply,fc_reply,num_unique_langs,pct_geo_tweets,has_reply_inconsistency,reply_inconsistency_count
0,u1000050990,"[{'id': 't1491853123429847048', 'text': '@anto...",240,1,0.070609,0.15,0.15,0.70,1045.770048,95.325000,...,951.787500,0.029167,0.000000,955.841667,0.0,0.0,7,0.0,1,85
1,u1000054188183904257,"[{'id': 't1491355901462474753', 'text': 'RT @Z...",191,1,0.135846,0.00,0.05,0.95,3496.520817,94.952880,...,52.816754,0.005236,0.000000,53.151832,0.0,0.0,7,0.0,1,108
2,u100008784,"[{'id': 't1499241032160096256', 'text': 'My ty...",47,1,0.010500,0.15,0.40,0.45,50.535970,119.127660,...,2.170213,0.000000,0.000000,19.914894,0.0,0.0,2,0.0,1,31
3,u1000156482690838530,"[{'id': 't1491545672713187332', 'text': 'Jenni...",206,1,0.146515,0.40,0.05,0.55,18.559352,150.053398,...,5.621359,0.000000,0.000000,10.441748,0.0,0.0,4,0.0,1,3
4,u1000230050736701441,"[{'id': 't1492039865315500034', 'text': '@Spik...",377,1,0.268327,0.65,0.15,0.20,7.711959,70.602122,...,3.355438,0.673740,0.119363,30.872679,0.0,0.0,2,0.0,1,344


In [ ]:
df_tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38771 entries, 0 to 38770
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           38771 non-null  string 
 1   tweets                       38771 non-null  object 
 2   num_tweets                   38771 non-null  int64  
 3   has_tweets                   38771 non-null  int64  
 4   tweets_frequency             38771 non-null  float64
 5   avg_positive                 38771 non-null  float64
 6   avg_negative                 38771 non-null  float64
 7   avg_neutral                  38771 non-null  float64
 8   posting_regularity           38771 non-null  float64
 9   avg_length_per_tweet         38771 non-null  float64
 10  avg_words_per_tweet          38771 non-null  float64
 11  avg_word_length_per_tweet    38771 non-null  float64
 12  avg_uppercase_pct            38771 non-null  float64
 13  avg_digit_pct   

In [ ]:
non_numeric_cols = ['id', 'tweets', 'has_tweets', 'overuse_hashtags', 'has_reply_inconsistency']
numeric_cols = df_tweets.columns.difference(non_numeric_cols)

df_tweets[numeric_cols].describe()

,avg_attachments_per_tweet,avg_digit_pct,avg_emoji,avg_hashtags_per_tweet,avg_length_per_tweet,avg_likes_per_tweet,avg_mentions_per_tweet,avg_negative,avg_neutral,avg_originals,...,num_unique_langs,pct_allow_reply,pct_geo_tweets,pct_tweets_with_attachments,pct_tweets_with_hashtags,pct_tweets_with_mentions,pct_tweets_with_urls,posting_regularity,reply_inconsistency_count,tweets_frequency
count,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,...,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,3.877100e+04,38771.000000,38771.000000
mean,0.523384,0.019969,0.451097,0.834069,135.087879,350.805760,1.167167,0.118057,0.590157,0.667363,...,4.563772,0.000402,0.003204,0.434618,0.270845,0.688304,0.304333,4.136739e+03,47.931959,0.182848
std,0.683104,0.014417,0.817337,1.824053,48.785222,3530.888291,1.143549,0.151581,0.240335,0.297155,...,3.474752,0.006093,0.034423,0.494109,0.282937,0.277633,0.270209,3.802260e+04,65.849371,0.413940
min,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000199
25%,0.000000,0.012459,0.059274,0.063830,103.985579,0.632431,0.754951,0.000000,0.400000,0.443396,...,2.000000,0.000000,0.000000,0.000000,0.047619,0.545455,0.100000,5.373636e+00,3.000000,0.037823
50%,0.000000,0.016738,0.211538,0.239796,129.000000,3.042345,1.094737,0.050000,0.550000,0.738095,...,4.000000,0.000000,0.000000,0.000000,0.159091,0.765432,0.222603,1.718397e+02,20.000000,0.071429
75%,1.000000,0.023143,0.555556,0.792266,159.827558,19.701913,1.427386,0.200000,0.750000,0.942308,...,6.000000,0.000000,0.000000,1.000000,0.421053,0.907162,0.428571,1.445333e+03,66.000000,0.175518
max,4.000000,0.590909,37.549020,31.291304,755.016393,246700.856287,48.147541,1.000000,1.000000,1.000000,...,35.000000,0.600000,1.000000,1.000000,1.000000,1.000000,1.000000,3.641385e+06,995.000000,14.857143


In [ ]:
print("has_tweets counts:")
print(df_tweets['has_tweets'].value_counts())

print("\noveruse_hashtags counts:")
print(df_tweets['overuse_hashtags'].value_counts())

print("\nhas_reply_inconsistency counts:")
print(df_tweets['has_reply_inconsistency'].value_counts())

has_tweets counts:
has_tweets
1    38771
Name: count, dtype: int64

overuse_hashtags counts:
overuse_hashtags
1    29903
0     8868
Name: count, dtype: int64

has_reply_inconsistency counts:
has_reply_inconsistency
1    33262
0     5509
Name: count, dtype: int64


## Export Features

In [38]:
df_tweets_model = df_tweets.drop(columns=["tweets"])
df_tweets_model.to_parquet("./datasets/final_outputs/df_tweets_model.parquet", index=False)

print(df_tweets_model.shape)
print(f"{df_tweets_model.memory_usage(deep=True).sum() / 1e6:,.1f} MB")

display(df_tweets_model[numeric_cols].describe())
del df_tweets_model
gc.collect()

(38771, 40)
14.5 MB


,avg_attachments_per_tweet,avg_digit_pct,avg_emoji,avg_hashtags_per_tweet,avg_length_per_tweet,avg_likes_per_tweet,avg_mentions_per_tweet,avg_negative,avg_neutral,avg_originals,...,num_unique_langs,pct_allow_reply,pct_geo_tweets,pct_tweets_with_attachments,pct_tweets_with_hashtags,pct_tweets_with_mentions,pct_tweets_with_urls,posting_regularity,reply_inconsistency_count,tweets_frequency
count,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,...,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,38771.000000,3.877100e+04,38771.000000,38771.000000
mean,0.523384,0.019969,0.451097,0.834069,135.087879,350.805760,1.167167,0.118057,0.590157,0.667363,...,4.563772,0.000402,0.003204,0.434618,0.270845,0.688304,0.304333,4.136739e+03,47.931959,0.182848
std,0.683104,0.014417,0.817337,1.824053,48.785222,3530.888291,1.143549,0.151581,0.240335,0.297155,...,3.474752,0.006093,0.034423,0.494109,0.282937,0.277633,0.270209,3.802260e+04,65.849371,0.413940
min,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000199
25%,0.000000,0.012459,0.059274,0.063830,103.985579,0.632431,0.754951,0.000000,0.400000,0.443396,...,2.000000,0.000000,0.000000,0.000000,0.047619,0.545455,0.100000,5.373636e+00,3.000000,0.037823
50%,0.000000,0.016738,0.211538,0.239796,129.000000,3.042345,1.094737,0.050000,0.550000,0.738095,...,4.000000,0.000000,0.000000,0.000000,0.159091,0.765432,0.222603,1.718397e+02,20.000000,0.071429
75%,1.000000,0.023143,0.555556,0.792266,159.827558,19.701913,1.427386,0.200000,0.750000,0.942308,...,6.000000,0.000000,0.000000,1.000000,0.421053,0.907162,0.428571,1.445333e+03,66.000000,0.175518
max,4.000000,0.590909,37.549020,31.291304,755.016393,246700.856287,48.147541,1.000000,1.000000,1.000000,...,35.000000,0.600000,1.000000,1.000000,1.000000,1.000000,1.000000,3.641385e+06,995.000000,14.857143


13

In [ ]:
FEATURE_COLS = [c for c in df_tweets.columns if c not in ("id", "tweets")]

desc = df_tweets[FEATURE_COLS].describe().T
constant = desc[desc["std"] == 0]
print("=== CONSTANT FEATURES (std == 0) ===")
print(constant[["mean", "min", "max"]] if len(constant) else "none")
print()

all_zero = [c for c in FEATURE_COLS if (df_tweets[c] == 0).all()]
print("=== ALL-ZERO FEATURES ===")
print(all_zero if all_zero else "none")
print()

was_broken = [
    "avg_retweets", "avg_quotes", "avg_replies", "avg_originals",
    "avg_hashtags_per_tweet", "pct_tweets_with_hashtags",
    "avg_mentions_per_tweet", "pct_tweets_with_mentions",
    "avg_urls_per_tweet", "pct_tweets_with_urls",
    "avg_attachments_per_tweet", "pct_tweets_with_attachments",
    "reply_inconsistency_count",
]
print("=== PREVIOUSLY BROKEN FEATURES ===")
print(df_tweets[[c for c in was_broken if c in df_tweets.columns]].describe().T[["mean", "std", "max"]])
print()

type_cols = ["avg_retweets", "avg_quotes", "avg_replies", "avg_originals"]
print("=== TWEET TYPE SUM ===")
print("Sum of means:", df_tweets[type_cols].mean().sum().round(4))
print("(slightly above 1.0 expected; a tweet can be both reply and quote)")
print()

nans = df_tweets[FEATURE_COLS].isna().sum()
print("=== NaN COUNTS ===")
print(nans[nans > 0] if nans.sum() else "none")

import numpy as np
numeric = df_tweets[FEATURE_COLS].select_dtypes("number")
infs = np.isinf(numeric).sum()
print("=== INF COUNTS ===")
print(infs[infs > 0] if infs.sum() else "none")
print()

sampled = set(df_users["id"])
covered = set(df_tweets["id"])
print("=== COVERAGE ===")
print(f"Sampled users:        {len(sampled):,}")
print(f"Users with tweets:    {len(covered):,}")
print(f"Users with no tweets: {len(sampled - covered):,}")
print(f"Ids not in sample:    {len(covered - sampled):,}")
print(f"Duplicate ids:        {df_tweets['id'].duplicated().sum()}")

=== CONSTANT FEATURES (std == 0) ===
            mean  min  max
has_tweets   1.0  1.0  1.0

=== ALL-ZERO FEATURES ===
none

=== PREVIOUSLY BROKEN FEATURES ===
                                  mean        std         max
avg_retweets                  0.039260   0.133630    1.000000
avg_quotes                    0.007696   0.030388    1.000000
avg_replies                   0.286255   0.271339    1.000000
avg_originals                 0.667363   0.297155    1.000000
avg_hashtags_per_tweet        0.834069   1.824053   31.291304
pct_tweets_with_hashtags      0.270845   0.282937    1.000000
avg_mentions_per_tweet        1.167167   1.143549   48.147541
pct_tweets_with_mentions      0.688304   0.277633    1.000000
avg_urls_per_tweet            0.333234   0.316575    4.555000
pct_tweets_with_urls          0.304333   0.270209    1.000000
avg_attachments_per_tweet     0.523384   0.683104    4.000000
pct_tweets_with_attachments   0.434618   0.494109    1.000000
reply_inconsistency_count    47.931